# Preprocesamiento de Datos

Transforma el dataset crudo extraido de los boletines en un dataset listo para entrenar modelos de IA.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
def preprocesar_datos(df):
    estadisticas = {
        "registros_originales": len(df),
        "productos_originales": df["producto_raw"].nunique(),
        "provincias": df["provincia"].nunique(),
    }

    df_limpio = df[df["estado_precio"] == "completo"].copy()
    estadisticas["registros_completos"] = len(df_limpio)
    estadisticas["registros_parciales"] = len(df[df["estado_precio"] == "parcial"])
    estadisticas["registros_invalidos"] = len(df[df["estado_precio"] == "invalido"])

    if "quincena_id" in df_limpio.columns:
        df_limpio["periodo"] = df_limpio["quincena_id"]
    elif "año" in df_limpio.columns and "quincena" in df_limpio.columns:
        df_limpio["periodo"] = df_limpio["año"].astype(str) + "-" + df_limpio["quincena"].astype(str).str.zfill(2)
    else:
        df_limpio["periodo"] = "desconocido"

    df_pivot = df_limpio.pivot_table(
        index=["producto_raw", "provincia"],
        columns="periodo",
        values=["precio_anterior", "precio_actual"],
        aggfunc="first"
    )

    df_pivot.columns = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
    df_pivot = df_pivot.reset_index()

    periodos_unicos = sorted(df_limpio["periodo"].unique())
    cols_ordenadas = ["producto_raw", "provincia"]
    for p in periodos_unicos:
        col_ant = f"precio_anterior_{p}"
        col_act = f"precio_actual_{p}"
        if col_ant in df_pivot.columns:
            cols_ordenadas.append(col_ant)
        if col_act in df_pivot.columns:
            cols_ordenadas.append(col_act)

    cols_existentes = [c for c in cols_ordenadas if c in df_pivot.columns]
    df_pivot = df_pivot[cols_existentes]

    productos_descartados = []
    productos_mantener = []

    for idx, row in df_pivot.iterrows():
        cols_precio = [c for c in df_pivot.columns if c.startswith("precio_actual_")]
        valores = row[cols_precio]
        total_periodos = len(valores)
        faltantes = valores.isna().sum()
        porcentaje_faltantes = faltantes / total_periodos if total_periodos > 0 else 1

        if porcentaje_faltantes > 0.30:
            productos_descartados.append({
                "producto": row["producto_raw"],
                "provincia": row["provincia"],
                "porcentaje_faltantes": round(porcentaje_faltantes * 100, 1)
            })
        else:
            productos_mantener.append(idx)

    df_pivot = df_pivot.loc[productos_mantener]

    cols_precio = [c for c in df_pivot.columns if c.startswith("precio_")]
    df_pivot[cols_precio] = df_pivot[cols_precio].interpolate(method="linear", axis=1)
    df_pivot[cols_precio] = df_pivot[cols_precio].ffill(axis=1).bfill(axis=1)

    estadisticas["registros_despues_filtro"] = len(df_pivot)
    estadisticas["productos_descartados"] = len(productos_descartados)
    estadisticas["lista_descartados"] = productos_descartados

    df_modelo, le_producto, le_provincia = _crear_features(df_pivot, periodos_unicos)

    estadisticas["registros_modelo"] = len(df_modelo)
    estadisticas["columnas_modelo"] = list(df_modelo.columns)

    return {
        "dataset_final": df_modelo,
        "dataset_wide": df_pivot,
        "estadisticas": estadisticas,
        "productos_descartados": productos_descartados,
        "le_producto": le_producto,
        "le_provincia": le_provincia,
    }

In [ ]:
def _crear_features(df_pivot, periodos_unicos):
    
    registros = []

    for idx, row in df_pivot.iterrows():
        producto = row["producto_raw"]
        provincia = row["provincia"]

        # Construir serie de precios temporal ordenada por este producto+provincia
        serie_precios = []
        for periodo in periodos_unicos:
            col_actual = f"precio_actual_{periodo}"
            if col_actual in df_pivot.columns:
                serie_precios.append((periodo, row.get(col_actual)))
            else:
                serie_precios.append((periodo, np.nan))

        # Calcular promedios moviles con rolling sobre la serie temporal
        precios_series = pd.Series([p[1] for p in serie_precios], index=[p[0] for p in serie_precios])
        pm2 = precios_series.shift(1).rolling(window=2, min_periods=2).mean()
        pm3 = precios_series.shift(1).rolling(window=3, min_periods=3).mean()

        for i, periodo in enumerate(periodos_unicos):
            col_actual = f"precio_actual_{periodo}"
            if col_actual not in df_pivot.columns:
                continue

            precio_actual = row.get(col_actual)
            if pd.isna(precio_actual):
                continue

            precio_t1 = None
            if i > 0:
                col_t1 = f"precio_actual_{periodos_unicos[i-1]}"
                if col_t1 in df_pivot.columns:
                    precio_t1 = row.get(col_t1)

            precio_t2 = None
            if i > 1:
                col_t2 = f"precio_actual_{periodos_unicos[i-2]}"
                if col_t2 in df_pivot.columns:
                    precio_t2 = row.get(col_t2)

            if precio_t1 is not None and not pd.isna(precio_t1):
                registro = {
                    "producto": producto,
                    "provincia": provincia,
                    "periodo": periodo,
                    "precio_actual": precio_actual,
                    "precio_t1": precio_t1,
                    "precio_t2": precio_t2 if precio_t2 is not None else np.nan,
                }

                if precio_t2 is not None and not pd.isna(precio_t2) and precio_t2 > 0:
                    registro["variacion_t2_t1"] = round(((precio_t1 - precio_t2) / precio_t2) * 100, 2)
                else:
                    registro["variacion_t2_t1"] = np.nan

                registro["promedio_movil_2q"] = round(pm2.loc[periodo], 4) if pd.notna(pm2.loc[periodo]) else np.nan
                registro["promedio_movil_3q"] = round(pm3.loc[periodo], 4) if pd.notna(pm3.loc[periodo]) else np.nan

                try:
                    partes = periodo.split("-")
                    if len(partes) == 3 and partes[2].startswith("Q"):
                        registro["año"] = int(partes[0])
                        registro["mes"] = int(partes[1])
                        registro["quincena"] = int(partes[2][1:])
                    else:
                        registro["año"] = int(partes[0])
                        registro["quincena"] = int(partes[1])
                        registro["mes"] = 1 if int(partes[1]) == 1 else 2
                except (ValueError, IndexError):
                    registro["mes"] = np.nan
                    registro["año"] = np.nan
                    registro["quincena"] = np.nan

                variacion_precio = ((precio_actual - precio_t1) / precio_t1) * 100 if precio_t1 > 0 else 0
                registro["variacion_real"] = round(variacion_precio, 2)

                if variacion_precio > 7:
                    registro["comportamiento"] = "Alza"
                elif variacion_precio < -7:
                    registro["comportamiento"] = "Caída"
                else:
                    registro["comportamiento"] = "Estable"

                registros.append(registro)

    df_modelo = pd.DataFrame(registros)
    df_modelo, le_producto, le_provincia = _codificar_categoricas(df_modelo)
    return df_modelo, le_producto, le_provincia

In [4]:
def _codificar_categoricas(df):
    le_producto = LabelEncoder()
    le_provincia = LabelEncoder()
    df["producto_encoded"] = le_producto.fit_transform(df["producto"])
    df["provincia_encoded"] = le_provincia.fit_transform(df["provincia"])
    return df, le_producto, le_provincia

In [5]:
def obtener_resumen(df_modelo):
    resumen = {
        "total_registros": len(df_modelo),
        "productos_unicos": df_modelo["producto"].nunique(),
        "provincias": df_modelo["provincia"].nunique(),
        "periodos": df_modelo["periodo"].nunique(),
        "distribucion_comportamiento": df_modelo["comportamiento"].value_counts().to_dict(),
        "valores_faltantes": df_modelo.isnull().sum().to_dict(),
    }
    return resumen

In [6]:
%whos DataFrame

No variables match your requested type.


In [7]:
import pandas as pd

# Cargar el dataset crudo (ajusta la ruta si es necesario)
df_crudo = pd.read_csv('data/processed/dataset_crudo_sipa.csv')

# Ejecutar el preprocesamiento real
resultado = preprocesar_datos(df_crudo)
df_modelo = resultado['dataset_final']

print(f"Registros en el dataset final: {len(df_modelo)}")
df_modelo.head()

Registros en el dataset final: 1584


,producto,provincia,periodo,precio_actual,precio_t1,precio_t2,variacion_t2_t1,promedio_movil_2q,promedio_movil_3q,año,mes,quincena,variacion_real,comportamiento,producto_encoded,provincia_encoded
0,Aceite Vegetal - Favorita (Caja aprox. 15 lt),AZUAY,2026-01-Q2,33.0,33.0,NaN,NaN,NaN,NaN,2026,1,2,0.0,Estable,0,0
1,Aceite Vegetal - Favorita (Caja aprox. 15 lt),AZUAY,2026-02-Q1,33.0,33.0,33.0,0.0,33.0,NaN,2026,2,1,0.0,Estable,0,0
2,Aceite Vegetal - Favorita (Caja aprox. 15 lt),AZUAY,2026-02-Q2,33.0,33.0,33.0,0.0,33.0,33.0,2026,2,2,0.0,Estable,0,0
3,Aceite Vegetal - Favorita (Caja aprox. 15 lt),AZUAY,2026-03-Q1,33.0,33.0,33.0,0.0,33.0,33.0,2026,3,1,0.0,Estable,0,0
4,Aceite Vegetal - Favorita (Caja aprox. 15 lt),AZUAY,2026-03-Q2,33.0,33.0,33.0,0.0,33.0,33.0,2026,3,2,0.0,Estable,0,0


In [8]:
print(df_modelo['comportamiento'].value_counts())
print()
print(df_modelo['comportamiento'].value_counts(normalize=True).round(3) * 100)

comportamiento
Estable    818
Caída      384
Alza       382
Name: count, dtype: int64

comportamiento
Estable    51.6
Caída      24.2
Alza       24.1
Name: proportion, dtype: float64


In [9]:
print(df_crudo['quincena_id'].nunique())
print(sorted(df_crudo['quincena_id'].unique()))

13
['2026-01-Q1', '2026-01-Q2', '2026-02-Q1', '2026-02-Q2', '2026-03-Q1', '2026-03-Q2', '2026-04-Q1', '2026-04-Q2', '2026-05-Q1', '2026-05-Q2', '2026-06-Q1', '2026-06-Q2', '2026-07-Q1']


In [11]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)